In [11]:
# PyTorch functions/methods helpers

# 8.4.4
branches = [torch.full((1, 2, 3, 3), fill_value=v) for v in [1.0, 2.0, 3.0]] # torch.full() creates a tensor of a specified shape where every element has the same value

* GoogLeNet uses Inception blocks: **several branches process the same input at different receptive-field scales, then concatenate their output channels**.

* The architecture lesson is that a block can be a small directed computation graph, not only a straight sequence.

# How to use this notebook

* Run the notebook from top to bottom in a clean kernel.

* The code uses small synthetic tensors so that architecture mechanics can be inspected without downloads, `torchvision`, ImageNet-scale images, or long training runs.

* Before important cells, predict the shape, parameter count, or failure mode, then read the assertions as executable contracts.

In [ ]:
import math

import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

def shape(x):
    return tuple(x.shape)

def count_parameters(module):
    return sum(p.numle() for p in module.parameters())

def trace_module_shapes(module, X):
    rows = []
    current = X
    for name, layer in module.named_children():
        current = layer(current)
        rows.append((name, layer.__class__.__name__, shape(current)))
    return rows, current

# 8.4.0 The Problem This Notebook Solves

VGG repeats one simple path. GoogLeNet asks a different question:

```text
what if useful features need different receptive-field sizes at the same network depth?
```

An Inception block answers with branches:

- a 1 by 1 branch
- a 1 by 1 then 3 by 3 branch
- a 1 by 1 then 5 by 5 branch
- a pooling then 1 by 1 branch

The branch outputs are concatenated along the channel dimension.

Concatenation means the next layer receives all branch feature maps as one wider tensor.

# 8.4.1 An Inception Block Concatenates Channel Evidence

Each branch must output the same height and width.

The channel counts can differ because concatenation happens along the channel axis.

Before running the cell, predict:

- Branch spatial shapes should all be `16 by 16`.
- Output channels should be `4 + 6 + 6 + 4 = 20`.
- The output tensor should have shape `(2, 20, 16, 16)`.

In [ ]:
class Inception(nn.Module):

    def __init__(self, in_channels, c1, c2, c3, c4):

        super().__init__()

        self.b1 = nn.Sequential(nn.Conv2d(in_channels, c1, kernel_size=1), nn.ReLU())

        self.b2 = nn.Sequential(nn.Conv2d(in_channels, c2[0], kernel_size=1), nn.ReLU(),
                                nn.Conv2d(c2[0], c2[1], kernel_size=3, padding=1), nn.ReLU())

        self.b3 = nn.Sequential(nn.Conv2d(in_channels, c3[0], kernel_size=1), nn.ReLU(),
                                nn.Conv2d(c3[0], c3[1], kernel_size=5, padding=2), nn.ReLU())

        self.b4 = nn.Sequential(nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
                                nn.Conv2d(in_channels, c4, kernel_size=1), nn.ReLU())


    def forward(self, X):
        branches = [self.b1(X), self.b2(X), self.b3(X), self.b4(X)]
        self.last_branch_shapes = [shape(branch) for branch in branches]
        return torch.cat(branches, dim=1)

block = Inception(3, c1=4, c2=(4, 6), c3=(4, 6), c4=4)
Y = block(torch.randn(2, 3, 16, 16))

# self.b1 = output size of 16 - 1 + 1 = 16 for shape of (batch, out_channels, height, width) or (2, 4, 16, 16)
# self.b2 = 16 - 1 + 1 = 16, for (2, 4, 16, 16) -> 16 + 2*1 - 3 + 1 = 16, for (2, 6, 16, 16)
# self.b3 = 16 - 1 + 1 = 16, for (2, 4, 16, 16) -> 16 + 2*2 - 5 + 1 = 16, for (2, 6, 16, 16)
# self.b4 = 16 + 2*1 - 3 + 1 = 16, for (2, 4, 16, 16) -> 16 - 1 + 1 = 16, for (2, 4, 16, 16)
# torch.cat(branches, dim=1) = 4 + 6 + 6 + 4 = 20 for (2, 20, 16, 16)

print("branch shapes:", block.last_branch_shapes)
print("output:", shape(Y))
assert shape(Y) == (2, 20, 16, 16)

branch shapes: [(2, 4, 16, 16), (2, 6, 16, 16), (2, 6, 16, 16), (2, 4, 16, 16)]
output: (2, 20, 16, 16)


# 8.4.2 Bottleneck 1 by 1 Convolutions Reduce Large-Kernel Cost

A 5 by 5 convolution from many input channels to many output channels (20 shown above) is expensive.

Inception often inserts a 1 by 1 convolution first to reduce channel width:

```text
many channels -> fewer channels -> 5 by 5 convolution
```

This is called a bottleneck because information passes through a narrower channel dimension before the expensive operation.

In [ ]:
in_channels = 64
reduced_channels = 16
out_channels = 32

direct_5x5 = in_channels * out_channels * 5 * 5                   # 64 * 32 * 5 * 5
bottleneck_then_5x5 = in_channels * reduced_channels * 1 * 1      # 64 * 16 * 1 * 1 (bottleneck), for example, nn.Conv2d(in_channels=32, out_channels=16, kernel_size=1)
bottleneck_then_5x5 += reduced_channels * out_channels * 5 * 5    # 64 * 16 * 1 * 1 (bottleneck) + 16 * 32 * 5 * 5 (5x5, where the out_channels in bottleneck becomes in_channels in 5x5)

print("direct 5x5 weights:", direct_5x5)                          # 51,200
print("1x1 bottleneck then 5x5 weights:", bottleneck_then_5x5)    # 13,824

assert bottleneck_then_5x5 < direct_5x5

direct 5x5 weights: 51200
1x1 bottleneck then 5x5 weights: 13824


# 8.4.3 Build a Tiny GoogLeNet-Style Classifier

The full GoogLeNet has many Inception blocks. This notebook uses a tiny version that preserves the design contract:

```text
stem -> Inception block -> pooling -> Inception block -> global average pool -> logits
```

The first Inception block receives 8 channels and outputs 16. After pooling, the second receives 16 and outputs 32.

In [ ]:
googlenet_tiny = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=3, padding=1), nn.ReLU(),
    Inception(8, c1=4, c2=(4, 4), c3=(4, 4), c4=4),
    nn.MaxPool2d(kernel_size=2, stride=2),
    Inception(16, c1=8, c2=(8, 8), c3=(8, 8), c4=8),
    nn.AdaptiveAvgPool2d((1, 1)),
    nn.Flatten(),
    nn.Linear(32, 10)
)

X = torch.randn(2, 1, 32, 32)

# Layer 0 = output size of floor((32 + 2*1 - 3) + 1) = 32 for shape of (batch, out_channels, height, width) or (2, 8, 32, 32)
# Layer 1 = ReLU
# Layer 2 = out_channels of 3 + 4 + 4 + 4 = 15 for shape of (batch, out_channels, height, width) or (2, 16, 32, 32)
# Layer 3 = output size of floor((32 - 2)/2 + 1) = 16 for shape of (batch, out_channels, height, width) or (2, 16, 16, 16)
# Layer 4 = out_channels of 16 + 8 + 8 + 8 = 32 for shape of (batch, out_channels, height, width) or (2, 32, 16, 16)
# Layer 5 = (2, 32, 1, 1) as height and width are now averaged per dimension (across columns)
# Layer 6 = (2, 32)
# Layer 7 = Layer 5 shape of (2, 32) @ linear.weight.T shape of (32, 10) = final output shape of (2, 10)

rows, logits = trace_module_shapes(googlenet_tiny, X)
for row in rows:
    print(row)

assert shape(logits) == (2, 10)

('0', 'Conv2d', (2, 8, 32, 32))
('1', 'ReLU', (2, 8, 32, 32))
('2', 'Inception', (2, 16, 32, 32))
('3', 'MaxPool2d', (2, 16, 16, 16))
('4', 'Inception', (2, 32, 16, 16))
('5', 'AdaptiveAvgPool2d', (2, 32, 1, 1))
('6', 'Flatten', (2, 32))
('7', 'Linear', (2, 10))


# 8.4.4 Concatenation Preserves Branch Identity as Channels

After concatenation, the next layer sees one tensor.

It does not know which branch each channel came from unless you track the channel ranges yourself.

That is the engineering discipline:

```text
branch 1 channels occupy one slice
branch 2 channels occupy the next slice
branch 3 channels occupy the next slice
branch 4 channels occupy the final slice
```

The cell manually verifies the channel slices after concatenation.

In [ ]:
branches = [torch.full((1, 2, 3, 3), fill_value=v) for v in [1.0, 2.0, 3.0]] # torch.full() creates a tensor of a specified shape where every element has the same value
merged = torch.cat(branches, dim=1) # Each branch has 2 channels, so 2*3 = 6 for shape of (batch, out_channels, height, width) or (1, 6, 3, 3)

print("merged shape:", shape(merged))
print("channel means:", merged.mean(dim=(0, 2, 3)))

assert shape(merged) == (1, 6, 3, 3)
assert torch.equal(merged[:, 0:2], branches[0])
assert torch.equal(merged[:, 2:4], branches[1])
assert torch.equal(merged[:, 4:6], branches[2])

merged shape: (1, 6, 3, 3)
channel means: tensor([1., 1., 2., 2., 3., 3.])


# 8.4.5 Break It Deliberately: Branch Spatial Mismatch

* Concatenation along channels only works when **every other dimension matches**.
* If one branch changes height or width, `torch.cat(..., dim=1)` rejects the result.

The theory-level mistake is asking the next layer to treat nonaligned spatial grids as if they were feature channels at the same locations.

In [ ]:
class BadInception(nn.Module):

    def __init__(self):
        super().__init__()
        self.good = nn.Conv2d(3, 4, kernel_size=1)                        # 16 - 1 + 1 = 16
        self.bad = nn.Conv2d(3, 4, kernel_size=3, stride=2, padding=1)    # floor(16 + 2*1 - 3)/2 + 1 = floor(8.5) = 8

    def forward(self, X):
        return torch.cat([self.good(X), self.bad(X)], dim=1)              # Would not work for self.bad since forward (torch.cat) expects (2, out_channels, 16, 16) yet receives (2, out_channels, 8, 8)

try:
    BadInception()(torch.randn(2, 3, 16, 16))
except RuntimeError as error:
    print(type(error).__name__)
    print(str(error).splitlines()[0])
else:
    raise AssertionError("Expected branch spatial mismatch to fail")

RuntimeError
Sizes of tensors must match except in dimension 1. Expected size 16 but got size 8 for tensor number 1 in the list.


# 8.4 Checkpoint

Answer these before moving on.

Short markdown answers in the notebook are enough; the checkpoint is meant to test whether you can explain the mechanics without rereading the code.

1. Why does an Inception block use several branches instead of one path?
> Because different branches can learn different features at different spatial scales, and their outputs can then be concatenated along the channel dimension

2. Why must branch outputs agree on height and width before concatenation?
> Because `torch.cat()` requires all other dimensions other than the concatenation dimensions to match

3. How does a 1 by 1 bottleneck reduce the cost of a 5 by 5 branch?
> A 1×1 convolution first reduces the number of channels entering the expensive 5×5 convolution, greatly reducing the number of parameters and computations

4. After concatenation, where is branch identity stored?
> Branch identity is implicit in the channel positions: each branch's output occupies a particular range of channels in the concatenated tensor

5. Why is an Inception block not naturally represented as a simple `nn.Sequential` inside its own forward logic?
> Because it requires a contacnetation logic, which must be performed at the end of all convolutions (e.g. it's not an optimization method like ReLU that you can add alongside the layers)